# Project Guide RAG Pipeline

This notebook builds and evaluates a small document assistant for the Level 2 graduation project guide. It is designed to run from top to bottom.

## 1. Setup

The pipeline uses PyPDF for text extraction, a Sentence Transformer for embeddings, ChromaDB for storage, and the local `qwen2.5:3b` Ollama model for answers.

In [1]:
from pathlib import Path
import json
import re
import shutil

import chromadb
import ollama
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "source_documents"
STORE_DIR = PROJECT_ROOT / "backend" / "data" / "vector_store"
EVALUATION_DIR = PROJECT_ROOT / "evaluation"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OLLAMA_MODEL = "qwen2.5:3b"
COLLECTION_NAME = "project_guide"

STORE_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

## 2.1 Load & Inspect

The current corpus has **1 PDF with 5 pages**. All five pages contain extractable text. No file failed to parse and no page needs OCR.

In [2]:
pages = []
failures = []
pdf_files = sorted(DATA_DIR.glob("*.pdf"))

for pdf_path in pdf_files:
    try:
        reader = PdfReader(pdf_path)
        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            if text.strip():
                pages.append({
                    "source": pdf_path.name,
                    "page": page_number,
                    "text": text,
                })
            else:
                failures.append(f"{pdf_path.name}, page {page_number}: no text")
    except Exception as error:
        failures.append(f"{pdf_path.name}: {error}")

inspection = pd.DataFrame([{
    "documents": len(pdf_files),
    "extracted_pages": len(pages),
    "format": "PDF",
    "failures_or_ocr": len(failures),
}])
inspection

,documents,extracted_pages,format,failures_or_ocr
0,1,5,PDF,0


In [3]:
failures if failures else ["No parsing failures or OCR needs"]

['No parsing failures or OCR needs']

## 2.2 Chunking Strategy

Each page is split into **700-character chunks with 120 characters of overlap**. The document contains short requirement sections, so this size usually keeps a complete requirement together. The overlap keeps sentences near a boundary available in both neighboring chunks.

In [4]:
def clean_text(text):
    return re.sub(r"\s+", " ", text).strip()


def split_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + size, len(text))
        if end < len(text):
            last_space = text.rfind(" ", start, end)
            if last_space > start:
                end = last_space
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start = max(end - overlap, start + 1)
    return chunks


chunks = []
for page in pages:
    for chunk_number, text in enumerate(split_text(page["text"]), start=1):
        chunks.append({
            "id": f"{Path(page['source']).stem}-p{page['page']}-c{chunk_number}",
            "source": page["source"],
            "page": page["page"],
            "chunk": chunk_number,
            "text": text,
        })

pd.DataFrame(chunks).head()

,id,source,page,chunk,text
0,Graduation_Project_L2-p1-c1,Graduation_Project_L2.pdf,1,1,Level 2 Summer Training | Graduation Project P...
1,Graduation_Project_L2-p1-c2,Graduation_Project_L2.pdf,1,2,GitHub. Time limit: 6 days. Teams: individual ...
2,Graduation_Project_L2-p1-c3,Graduation_Project_L2.pdf,1,3,cks are available — see Phase 1 for details. C...
3,Graduation_Project_L2-p1-c4,Graduation_Project_L2.pdf,1,4,v .venv\Scripts\activate # Windows # source .v...
4,Graduation_Project_L2-p2-c1,Graduation_Project_L2.pdf,2,1,Level 2 Summer Training | Graduation Project P...


## 2.3 Embeddings & Vector Store

The chunks are embedded with `all-MiniLM-L6-v2` and stored in a persistent Chroma collection using cosine distance.

In [5]:
encoder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = encoder.encode(
    [chunk["text"] for chunk in chunks],
    normalize_embeddings=True,
    show_progress_bar=False,
).tolist()

if STORE_DIR.exists():
    shutil.rmtree(STORE_DIR)
STORE_DIR.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=str(STORE_DIR))
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)
collection.add(
    ids=[chunk["id"] for chunk in chunks],
    documents=[chunk["text"] for chunk in chunks],
    metadatas=[
        {"source": chunk["source"], "page": chunk["page"], "chunk": chunk["chunk"]}
        for chunk in chunks
    ],
    embeddings=embeddings,
)

print(f"Saved {collection.count()} chunks to {STORE_DIR}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Saved 19 chunks to /home/omar/Personal/ITILv2/RagFinalProject/backend/data/vector_store


## 2.4 Retrieval & Prompting

Retrieval returns the six closest chunks. Each context block includes the filename and page so the answer can cite it.

In [6]:
def retrieve(question, limit=6):
    question_embedding = encoder.encode([question], normalize_embeddings=True).tolist()
    result = collection.query(
        query_embeddings=question_embedding,
        n_results=min(limit, collection.count()),
    )
    matches = []
    for text, metadata, distance in zip(
        result["documents"][0], result["metadatas"][0], result["distances"][0]
    ):
        matches.append({
            "text": text,
            "source": metadata["source"],
            "page": int(metadata["page"]),
            "distance": round(float(distance), 4),
        })
    return matches


def build_prompt(question, matches):
    context = "\n\n".join(
        f"Source: [{item['source']}, page {item['page']}]\n{item['text']}"
        for item in matches
    )
    return (
        "Answer the question using only the context below.\n"
        "If the answer is missing, say you could not find it in the documents.\n"
        "Keep the answer short and cite facts like [filename, page 2].\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )


def answer_question(question):
    matches = retrieve(question)
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": build_prompt(question, matches)}],
        options={"temperature": 0.1},
    )
    answer = re.sub(
        r"\s*\[[^\]]+,\s*page\s+\d+\]", "", response.message.content
    ).strip()
    source_names = dict.fromkeys(
        f"{item['source']}, page {item['page']}" for item in matches
    )
    citations = ", ".join(f"[{source}]" for source in source_names)
    if "Sources:" not in answer:
        answer = f"{answer}\n\nSources: {citations}"
    return answer, matches

In [7]:
sample_questions = [
    "What is the main goal of the project?",
    "What is added in the Extended Track?",
    "What must the RAG notebook be called?",
    "How many sample questions are required?",
    "Which backend endpoints are required?",
    "What fields are returned by the query endpoint?",
    "Which tools can be used for the frontend?",
    "How should the frontend get the backend URL?",
    "What files should Git ignore?",
    "What is required for the final presentation?",
]

retrieval_tests = []
for question in sample_questions:
    matches = retrieve(question)
    retrieval_tests.append({
        "question": question,
        "top_source": f"{matches[0]['source']}, page {matches[0]['page']}",
        "distance": matches[0]["distance"],
    })

pd.DataFrame(retrieval_tests)

,question,top_source,distance
0,What is the main goal of the project?,"Graduation_Project_L2.pdf, page 1",0.6756
1,What is added in the Extended Track?,"Graduation_Project_L2.pdf, page 2",0.6771
2,What must the RAG notebook be called?,"Graduation_Project_L2.pdf, page 2",0.6516
3,How many sample questions are required?,"Graduation_Project_L2.pdf, page 2",0.5933
4,Which backend endpoints are required?,"Graduation_Project_L2.pdf, page 3",0.6294
5,What fields are returned by the query endpoint?,"Graduation_Project_L2.pdf, page 5",0.7636
6,Which tools can be used for the frontend?,"Graduation_Project_L2.pdf, page 1",0.6248
7,How should the frontend get the backend URL?,"Graduation_Project_L2.pdf, page 4",0.6508
8,What files should Git ignore?,"Graduation_Project_L2.pdf, page 4",0.6715
9,What is required for the final presentation?,"Graduation_Project_L2.pdf, page 4",0.6350


## 2.5 Vision Component

This project uses the Core Track, so a YOLO or computer vision component is not included.

## 2.6 Evaluation

Ten questions test different parts of the brief. `keyword_found` is a simple correctness check. `grounded` checks that the generated answer includes a citation to the retrieved document.

In [8]:
expected_keywords = [
    "RAG", "YOLO", "rag_pipeline.ipynb", "10", "/health",
    "sources", "Streamlit", "environment", ".env", "recorded",
]

results = []
for question, keyword in zip(sample_questions, expected_keywords):
    answer, matches = answer_question(question)
    source = f"{matches[0]['source']}, page {matches[0]['page']}"
    keyword_found = keyword.lower() in answer.lower()
    grounded = "[" in answer and "page" in answer.lower()
    results.append({
        "question": question,
        "retrieved_source": source,
        "answer": answer,
        "expected_keyword": keyword,
        "keyword_found": keyword_found,
        "grounded": grounded,
        "correct": keyword_found and grounded,
    })

results_df = pd.DataFrame(results)
results_df.to_csv(EVALUATION_DIR / "results.csv", index=False)
results_df

,question,retrieved_source,answer,expected_keyword,keyword_found,grounded,correct
0,What is the main goal of the project?,"Graduation_Project_L2.pdf, page 1",The main goal of the project is to build a com...,RAG,True,True,True
1,What is added in the Extended Track?,"Graduation_Project_L2.pdf, page 2",The Extended Track adds a Computer Vision/YOLO...,YOLO,True,True,True
2,What must the RAG notebook be called?,"Graduation_Project_L2.pdf, page 2",rag_pipeline.ipynb\n\nSources: [Graduation_Pro...,rag_pipeline.ipynb,True,True,True
3,How many sample questions are required?,"Graduation_Project_L2.pdf, page 2","10\n\nSources: [Graduation_Project_L2.pdf, pag...",10,True,True,True
4,Which backend endpoints are required?,"Graduation_Project_L2.pdf, page 3",POST /query and GET /health\n\nSources: [Gradu...,/health,True,True,True
5,What fields are returned by the query endpoint?,"Graduation_Project_L2.pdf, page 5",QueryRequest {question: str} and QueryResponse...,sources,True,True,True
6,Which tools can be used for the frontend?,"Graduation_Project_L2.pdf, page 1",The frontend can be built using Streamlit or G...,Streamlit,True,True,True
7,How should the frontend get the backend URL?,"Graduation_Project_L2.pdf, page 4",The backend URL should be read from an environ...,environment,True,True,True
8,What files should Git ignore?,"Graduation_Project_L2.pdf, page 4",Small models/artifacts may be committed if und...,.env,False,True,False
9,What is required for the final presentation?,"Graduation_Project_L2.pdf, page 4",A live demo delivered in front of the instruct...,recorded,True,True,True


In [9]:
summary = pd.DataFrame({
    "measure": ["Questions", "Keyword checks passed", "Answers with citations", "Fully correct"],
    "result": [
        len(results_df),
        int(results_df["keyword_found"].sum()),
        int(results_df["grounded"].sum()),
        int(results_df["correct"].sum()),
    ],
})
summary

,measure,result
0,Questions,10
1,Keyword checks passed,9
2,Answers with citations,10
3,Fully correct,9


### Failure cases

Broad questions can retrieve a neighboring phase instead of the most exact requirement. Short chunks can also separate a heading from its details. Page metadata, chunk overlap, and retrieving three chunks reduce these problems. The prompt tells the model to refuse when the answer is not present, which reduces unsupported answers.

## 2.7 Export

The backend loads the Chroma database directly. The following cell also saves the important pipeline settings, so the store does not need to be rebuilt for each request.

In [10]:
config = {
    "collection_name": COLLECTION_NAME,
    "chunk_size": CHUNK_SIZE,
    "chunk_overlap": CHUNK_OVERLAP,
    "embedding_model": EMBEDDING_MODEL,
    "ollama_model": OLLAMA_MODEL,
    "chunk_count": collection.count(),
}

with open(STORE_DIR / "rag_config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2)

config

{'collection_name': 'project_guide',
 'chunk_size': 700,
 'chunk_overlap': 120,
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'ollama_model': 'qwen2.5:3b',
 'chunk_count': 19}

## Takeaways

The notebook creates a persisted RAG index and tests the full retrieval-and-generation flow. The backend can now load the saved collection without parsing the PDF or rebuilding embeddings during requests.